In [8]:
import pandas as pd

#datasets:---> 
df_new = pd.read_csv(r"C:\dscode\project_6th_sem\generative_ai_misinformation_dataset.csv")
df_old = pd.read_csv(r"C:\dscode\project_6th_sem\WELFake_Dataset.csv")
df_practice = pd.read_csv(r"C:\dscode\project_6th_sem\fake_news_dataset.csv")

def clean_and_standardize(df, name):
    #column-names_---remove acci...space and covert in  lower case
    df.columns = df.columns.str.strip().str.lower()    
    text_col = next((c for c in df.columns if c in ['text', 'body_text', 'content', 'body']), None)

    label_col = next((c for c in df.columns if c in ['label', 'is_misinformation', 'class', 'target']), None)
    
    if text_col and label_col:
        print(f"✅ {name}: Found columns '{text_col}' and '{label_col}'")
        new_df = df[[text_col, label_col]].copy()
        new_df.columns = ['text', 'label'] # Rename to standard
        return new_df
    else:
        print(f"❌ {name}: Could not find columns. Available: {df.columns.tolist()}")
        return None

#Clean all three dataset:--->
df_new_clean = clean_and_standardize(df_new, "Gen-AI Data")
df_old_clean = clean_and_standardize(df_old, "WELFake Data")
df_practice_clean = clean_and_standardize(df_practice, "Practice Data")

# Step 3: Combine only the ones that were cleaned successfully
valid_dfs = [d for d in [df_new_clean, df_old_clean, df_practice_clean] if d is not None]
master_df = pd.concat(valid_dfs, axis=0, ignore_index=True)

# Step 4: Final save
master_df.to_csv(r"C:\dscode\project_6th_sem\master_dataset.csv", index=False)
print(f"\n🚀 Success! Total rows in Master Dataset: {len(master_df)}")

✅ Gen-AI Data: Found columns 'text' and 'is_misinformation'
✅ WELFake Data: Found columns 'text' and 'label'
✅ Practice Data: Found columns 'text' and 'label'

🚀 Success! Total rows in Master Dataset: 92634


In [2]:
import pandas as pd

# read master data sett:-->?
df = pd.read_csv(r"C:\dscode\project_6th_sem\master_dataset.csv")
print(df['label'].value_counts())

label
1       37374
0       35260
fake    10056
real     9944
Name: count, dtype: int64


In [3]:
import pandas as pd
df = pd.read_csv(r"C:\dscode\project_6th_sem\master_dataset.csv")

df['label'] = df['label'].astype(str).str.lower().str.strip()
#map to 0 and 1:---/>,
# Assuming -------'label1' and 'real' are Real (1), and '0' and 'fake' are Fake (0)
mapping = {
    'label1': 1,
    'real': 1,
    '1': 1,
    'fake': 0,
    '0': 0
}

df['label'] = df['label'].map(mapping)
print("Missing values after mapping:", df['label'].isnull().sum())
df.dropna(subset=['label'], inplace=True)

#  check again----
print("\nFinal Unified Labels:")
print(df['label'].value_counts())
df.to_csv(r"C:\dscode\project_6th_sem\master_dataset_unified.csv", index=False)

Missing values after mapping: 0

Final Unified Labels:
label
1    47318
0    45316
Name: count, dtype: int64


In [4]:
import pandas as pd
df = pd.read_csv(r"C:\dscode\project_6th_sem\master_dataset_unified.csv")
df['label'] = df['label'].replace({'label1': 1})

df['label'] = df['label'].astype(int)
print("Final Unified Labels:")
print(df['label'].value_counts())
df.to_csv(r"C:\dscode\project_6th_sem\master_dataset_final.csv", index=False)


Final Unified Labels:
label
1    47318
0    45316
Name: count, dtype: int64


In [5]:
import re
import string
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def final_cleaning(text):
    text = str(text).lower() # Lowercase
    text = re.sub(r'\[.*?\]', '', text) # Remove text in brackets
    text = re.sub(r"\\W"," ",text) # Remove special characters
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remove URLs
    text = re.sub(r'<.*?>+', '', text) # Remove HTML
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text) # Punctuation
    text = re.sub(r'\n', '', text) # Remove newlines
    
    # Remove stopwords
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

# Apply to ....  dataset
print("Starting text cleaning... this might take 2-3 minutes.")
df['text'] = df['text'].apply(final_cleaning)

# Save the cleaned data
df.to_csv(r"C:\dscode\project_6th_sem\cleaned_data.csv", index=False)
print("✅ Done! Your data is now 100% ready for Machine Learning.")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\manga\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Starting text cleaning... this might take 2-3 minutes.
✅ Done! Your data is now 100% ready for Machine Learning.


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
df = pd.read_csv(r"C:\dscode\project_6th_sem\cleaned_data.csv")
df.dropna(inplace=True) # Ensure no empty rows remain

# 2. Split the data (80% for training, 20% for testing)
# x is the news text, y is the label (0 or 1);;
x_train, x_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=7)
# 3. Vectorization: Turn text into numbers using TF-IDF
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)
tfidf_train = tfidf_vectorizer.fit_transform(x_train) 
tfidf_test = tfidf_vectorizer.transform(x_test)

# 4. Initialize and Train the Model
pac = PassiveAggressiveClassifier(max_iter=50)
pac.fit(tfidf_train, y_train)

# 5. Test the Model
y_pred = pac.predict(tfidf_test)
score = accuracy_score(y_test, y_pred)
print(f'✅ Accuracy: {round(score*100,2)}%')

✅ Accuracy: 85.57%


In [7]:
import pickle

# Save the model
pickle.dump(pac, open(r"C:\dscode\project_6th_sem\models\fakenews_model.pkl", 'wb'))

# Save the vectorizer (VERY IMPORTANT: the web app needs this to understand new text)
pickle.dump(tfidf_vectorizer, open(r"C:\dscode\project_6th_sem\models\tfidf_vectorizer.pkl", 'wb'))

In [8]:

import re
import string

# 1. Load the Brain (Model and Vectorizer)
model = pickle.load(open(r"C:\dscode\project_6th_sem\models\fakenews_model.pkl", 'rb'))
vectorizer = pickle.load(open(r"C:\dscode\project_6th_sem\models\tfidf_vectorizer.pkl", 'rb'))

# 2. Re-use your cleaning function (Crucial: Input must be cleaned the same way!)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r"\\W"," ",text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[%s]' % re.escape(import streamlit as st
import picklestring.punctuation), '', text)
    return text

# 3. Build the UI
st.set_page_config(page_title="Fake News Detector", page_icon="🛡️")
st.title("🛡️ AI Fake News Detector")
st.markdown("Paste a news article below to check if it is **Real** or **Fake**.")

news_input = st.text_area("Enter News Text:", height=200)

if st.button("Analyze News"):
    if news_input:
        # Process the input
        cleaned_input = clean_text(news_input)
        vectorized_input = vectorizer.transform([cleaned_input])
        
        # Predict
        prediction = model.predict(vectorized_input)
        
        # Show Result
        if prediction[0] == 0:
            st.error("🚨 Result: This news is likely FAKE!")
        else:
            st.success("✅ Result: This news appears to be REAL.")
    else:
        st.warning("Please paste some text first.")

SyntaxError: invalid syntax (3896570625.py, line 14)